# 🧠 Análisis de Sentimientos con NLTK
### Procesamiento de Lenguaje Natural — Introducción práctica

---

## ¿Qué vamos a aprender hoy?

Imagina que tu jefe te pide revisar **500 comentarios** de clientes para saber si están contentos o molestos con el servicio. Leerlos uno por uno tomaría horas.

💡 **El análisis de sentimientos** es una técnica de NLP (Procesamiento de Lenguaje Natural) que le enseña a una computadora a identificar si un texto expresa una emoción **positiva**, **negativa** o **neutral**.

**Ejemplos del mundo real:**
- 📱 Apps que analizan reseñas en la tienda
- 🐦 Marcas que monitorean lo que dicen en redes sociales
- 🏥 Hospitales que evalúan la satisfacción de pacientes

---

---
## ⚙️ Sección 1: Configuración del entorno

Primero instalamos las librerías que vamos a usar. Solo necesitas ejecutar esta celda **una vez**.

In [1]:
# Instalamos las librerías necesarias
# %pip install nltk 
# %pip install pysentimiento

In [1]:
import nltk
import matplotlib.pyplot as plt
import pandas as pd

# Descargar los recursos de nltk
nltk.download('vader_lexicon') # diccionario vader con puntuaciones de sentimineto
nltk.download('punkt') # tokenizador: dividir el texto en palabras y oraciones
nltk.download('stopwords') # para palabras vacías


[nltk_data] Downloading package vader_lexicon to C:\Users\ANALISIS DE
[nltk_data]     DATOS\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package punkt to C:\Users\ANALISIS DE
[nltk_data]     DATOS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\ANALISIS DE
[nltk_data]     DATOS\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

---
Sección 2: Análisis en inglés con VADER

### ¿Qué es VADER?

**VADER** (Valence Aware Dictionary and sEntiment Reasoner) es una herramienta incluida en NLTK que analiza el sentimiento de un texto en inglés.

Fue diseñada especialmente para textos cortos como redes sociales, y entiende:
- Palabras en MAYÚSCULAS (énfasis)
- Signos de exclamación !!!!
- Emojis 😊😡

### ¿Qué devuelve VADER?

VADER devuelve 4 valores entre 0 y 1:

| Clave | Significado |
|-------|-------------|
| `pos` | Proporción de positividad |
| `neg` | Proporción de negatividad |
| `neu` | Proporción de neutralidad |
| `compound` | Puntaje global (de -1 a 1) |

> 💡 **El `compound` es el más importante:**
> - Mayor a **0.05** → Positivo ✅
> - Menor a **-0.05** → Negativo ❌
> - Entre ambos → Neutral ➖


texto = "I love this product! It is absolutely amazing."

In [2]:
# importar el analizador de intensidad de sentimientos
from nltk.sentiment import SentimentIntensityAnalyzer

# crear una instancia del analizador
analizador = SentimentIntensityAnalyzer()

# defjinimos el texto
texto = 'I love this product! It is absolutely amazing.'

# polarity scores, analizado el texto y devuelve un diccionario con: pos, neg, neu y compound
resultado = analizador.polarity_scores(texto)

# mostrar xada puntuacion
print(f"texto analizado: '{texto}'")
print(f'Resultados:')
print(f'Positivo: {resultado['pos']:.2f}')
print(f'Negativo: {resultado['neg']:.2f}')
print(f'Neutral: {resultado['neu']:.2f}')
print(f'Compound: {resultado['compound']:.2f}')

texto analizado: 'I love this product! It is absolutely amazing.'
Resultados:
Positivo: 0.63
Negativo: 0.00
Neutral: 0.37
Compound: 0.86


### 2.1 Convirtiendo el score en una etiqueta legible

Hagamos una función que nos diga si un texto es Positivo, Negativo o Neutral:  

ejemplos = [
    "This movie was absolutely fantastic! I loved every second.",
    "I hate this app. It keeps crashing and is totally useless.",
    "The delivery arrived on Tuesday.",
    "This is the WORST experience I have ever had!!!",
    "Not bad at all, actually pretty good 😊",
    "I don't like this at all 😡"
]

In [3]:
# definimos una función reutilizable 
def clasificar_sentimiento(texto):
    # analizar un texto y devolver su sentimiento
    scores = analizador.polarity_scores(texto)
    compound = scores['compound'] # uso solo del compound

    # clasificar de acuerdo a umbreales
    if compound >= 0.05:
        sentimiento = 'Positivo'
    elif compound <= -0.05:
        sentimiento = 'Negativo'
    else:
        sentimiento = 'Neutral'

    print(f'Texto: {texto}')
    print(f'-> Sentimiento {sentimiento} (compound: {compound:.2f})')
    print()

print('='*55)
print('PROBANDO CON DIFERENTES ORACIONES')
print('='*55)

ejemplos = [
    "This movie was absolutely fantastic! I loved every second.",
    "I hate this app. It keeps crashing and is totally useless.",
    "The delivery arrived on Tuesday.",
    "This is the WORST experience I have ever had!!!",
    "Not bad at all, actually pretty good 😊",
    "I don't like this at all 😡"
]

for texto in ejemplos:
    clasificar_sentimiento(texto)



PROBANDO CON DIFERENTES ORACIONES
Texto: This movie was absolutely fantastic! I loved every second.
-> Sentimiento Positivo (compound: 0.85)

Texto: I hate this app. It keeps crashing and is totally useless.
-> Sentimiento Negativo (compound: -0.78)

Texto: The delivery arrived on Tuesday.
-> Sentimiento Neutral (compound: 0.00)

Texto: This is the WORST experience I have ever had!!!
-> Sentimiento Negativo (compound: -0.77)

Texto: Not bad at all, actually pretty good 😊
-> Sentimiento Positivo (compound: 0.84)

Texto: I don't like this at all 😡
-> Sentimiento Negativo (compound: -0.28)



### 2.2 ¿Notaste algo interesante?

VADER entiende el contexto de varias formas:
- **MAYÚSCULAS** aumentan la intensidad del sentimiento
- Los **!!!** también aumentan la intensidad
- Los **emojis** tienen sentimiento asignado
- La palabra **"not"** puede invertir el sentimiento

Veamos esto con un ejemplo:

In [4]:
print('Efecto de las mayúsculas y signos de exclamación')
print()

variaciones = [
    'good',
    'GOOD',
    'good!',
    'GOOD!',
    'GOOD!!',
    'GOOD!!!',
]

for v in variaciones:
    score = analizador.polarity_scores(v)['compound']
    print(f"'{v}' -> compound: {score:.2f}")

Efecto de las mayúsculas y signos de exclamación

'good' -> compound: 0.44
'GOOD' -> compound: 0.44
'good!' -> compound: 0.49
'GOOD!' -> compound: 0.49
'GOOD!!' -> compound: 0.54
'GOOD!!!' -> compound: 0.58


---
## 🤖 Sección 3: Análisis en español con pysentimiento

### ¿Qué es pysentimiento?

**pysentimiento** es una librería open source que usa modelos de inteligencia artificial (transformers) entrenados con millones de textos en español.

A diferencia de VADER, que usa un diccionario de palabras con puntos, pysentimiento *entendió el idioma* leyendo muchísimos textos reales.

| | VADER | pysentimiento |
|---|---|---|
| Idioma principal | Inglés | Español (y más) |
| Cómo funciona | Diccionario de palabras | Modelo de IA (RoBERTa) |
| Velocidad | Muy rápido | Más lento (carga un modelo) |
| Precisión en español | Baja | Alta |

> 💡 **¿Qué devuelve?**  
> Una etiqueta (`POS`, `NEG`, `NEU`) y la probabilidad de cada una.


In [6]:
# %pip install pysentimiento

In [5]:
# este import nos permite cargar el modelo de pysentimiento
from pysentimiento import create_analyzer

analizador_es = create_analyzer(task='sentiment', lang='es')

texto = 'Me encanta este café, el servicio fue excelente!'

resultado = analizador_es.predict(texto)

print(f'Texto: {texto}')
print(f'\nResultados')
print(f'Etiqueta: {resultado.output}')
print(f'POS: {resultado.probas['POS']:.2f}')
print(f'NEG: {resultado.probas['NEG']:.2f}')
print(f'NEU: {resultado.probas['NEU']:.2f}')

model.safetensors:   0%|          | 0.00/435M [00:00<?, ?B/s]

c:\Users\ANALISIS DE DATOS\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ANALISIS DE DATOS\.cache\huggingface\hub\models--pysentimiento--robertuito-sentiment-analysis. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

Texto: Me encanta este café, el servicio fue excelente!

Resultados
Etiqueta: POS
POS: 0.98
NEG: 0.00
NEU: 0.02


### 3.1 Convirtiendo la etiqueta a texto legible

Igual que hicimos con VADER, creamos una función que muestre el resultado de forma clara.  
Ojo: La confianza no mide si algo es bueno o malo, mide qué tan seguro está el modelo de su clasificación.  

ejemplos_es = [
    "Este café en el centro histórico es maravilloso, lo recomiendo.",
    "Pésimo servicio, el café llegó frío y el mesero fue muy grosero.",
    "El café abrió a las 8 de la mañana el martes.",
    "INCREÍBLE vista desde la terraza! Sin duda el mejor lugar de Quito.",
    "No me gustó para nada, muy caro y sin sabor.",
    "El local es bonito pero el café es del montón.",
]

In [ ]:
def clasificar_con_pysentimiento(texto):
    resultado = analizador_es.predict(texto)
    etiqueta = resultado.output

    if etiqueta == 'POS':
        sentimiento = 'Positivo'
    elif etiqueta == 'NEG':
        sentimiento = 'Negativo'
    else:
        sentimiento = 'Neutro'

    prob = resultado.probas[etiqueta]
    print(f'Texto: {texto}')
    print(f'Sentimiento: {sentimiento} (confianza: {prob:.0%})')

print('='*55)
print('PROBANDO CON DIFERENTES ORACIONES')
print('='*55)

ejemplos_es = [
    "Este café en el centro histórico es maravilloso, lo recomiendo.",
    "Pésimo servicio, el café llegó frío y el mesero fue muy grosero.",
    "El café abrió a las 8 de la mañana el martes.",
    "INCREÍBLE vista desde la terraza! Sin duda el mejor lugar de Quito.",
    "No me gustó para nada, muy caro y sin sabor.",
    "El local es bonito pero el café es del montón.",
]

# clasificar
for texto in ejemplos_es:
    clasificar_con_pysentimiento(texto)

PROBANDO CON DIFERENTES ORACIONES
Texto: Este café en el centro histórico es maravilloso, lo recomiendo.
Sentimiento: Positivo (confianza: 98%)
Texto: Pésimo servicio, el café llegó frío y el mesero fue muy grosero.
Sentimiento: Negativo (confianza: 98%)
Texto: El café abrió a las 8 de la mañana el martes.
Sentimiento: Neutro (confianza: 89%)
Texto: INCREÍBLE vista desde la terraza! Sin duda el mejor lugar de Quito.
Sentimiento: Positivo (confianza: 98%)
Texto: No me gustó para nada, muy caro y sin sabor.
Sentimiento: Negativo (confianza: 94%)
Texto: El local es bonito pero el café es del montón.
Sentimiento: Neutro (confianza: 50%)


### 3.2 ¿Por qué pysentimiento es mejor para español?

Prueba esta misma oración con VADER y con pysentimiento y compara:


In [ ]:
texto_prueba = 'Que rico café, me encantó la atención!'

score_vader = analizador.polarity_scores(texto_prueba)['compound']

resultado_pys = analizador_es.predict(texto_prueba)

print(f'Texto: {texto_prueba}')
print()
print(f'VADER -> compound: {score_vader:.2f}')
print(f'PYSENTIMIENTO -> {resultado_pys.output}, confianza: {max(resultado_pys.probas.values()):.0%}')

Texto: Que rico café, me encantó la atención!

VADER -> compound: 0.00
PYSENTIMIENTO -> POS, confianza: 98%


---
Turno del Estudiante de Experimentar

Ahora es tu momento. Reutilizen código y prueben el analizador con sus propios textos.

### Ejercicio 1: Prueba con textos en inglés

In [16]:
for text in ejemplos:
    clasificar_con_pysentimiento(text)

Texto: This movie was absolutely fantastic! I loved every second.
Sentimiento: Positivo (confianza: 94%)
Texto: I hate this app. It keeps crashing and is totally useless.
Sentimiento: Negativo (confianza: 97%)
Texto: The delivery arrived on Tuesday.
Sentimiento: Neutro (confianza: 63%)
Texto: This is the WORST experience I have ever had!!!
Sentimiento: Negativo (confianza: 96%)
Texto: Not bad at all, actually pretty good 😊
Sentimiento: Positivo (confianza: 92%)
Texto: I don't like this at all 😡
Sentimiento: Negativo (confianza: 94%)


In [17]:
for text in ejemplos:
    clasificar_sentimiento(text)

Texto: This movie was absolutely fantastic! I loved every second.
-> Sentimiento Positivo (compound: 0.85)

Texto: I hate this app. It keeps crashing and is totally useless.
-> Sentimiento Negativo (compound: -0.78)

Texto: The delivery arrived on Tuesday.
-> Sentimiento Neutral (compound: 0.00)

Texto: This is the WORST experience I have ever had!!!
-> Sentimiento Negativo (compound: -0.77)

Texto: Not bad at all, actually pretty good 😊
-> Sentimiento Positivo (compound: 0.84)

Texto: I don't like this at all 😡
-> Sentimiento Negativo (compound: -0.28)



---
## 📝 Resumen de lo aprendido

| Concepto | ¿Qué aprendimos? |
|---|---|
| **Análisis de sentimientos** | Clasificar texto como positivo, negativo o neutral |
| **VADER** | Herramienta de NLTK para análisis en inglés |
| **Score compound** | El puntaje principal: va de -1 (muy negativo) a +1 (muy positivo) |
| **Analizador léxico** | Cómo funciona por dentro: diccionario + suma de puntos |
| **Limitaciones** | Las negaciones y el sarcasmo son retos reales del NLP |

Si quieres profundizar más en este tema, puedes explorar:

- **TextBlob** — librería más sencilla que incluye soporte básico para español
- **transformers (HuggingFace)** — modelos de IA avanzados preentrenados para español
- **scikit-learn** — para entrenar tu propio clasificador de sentimientos

---
